# Глава 7. Модель Prophet и перекрёстная проверка (`backtest`)

## 1. Теория

### 1.1 Идея Prophet

**Prophet** — модель, которая смотрит на ряд не как на цепочку «следующее значение из предыдущих», а как на **кривую, которую надо провести через точки**. Время — это просто ось X, а прогноз — та же кривая, продолженная вправо. Ряд раскладывается на сумму понятных слагаемых, и каждое подгоняется регрессией:

$$\LARGE y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

где:
- $\large y(t)$ — значение ряда в момент $\large t$ (заказы юнита за день).
- $\large g(t)$ — **тренд**: кусочно-линейная функция времени. Прямая, которая в некоторых точках может «переломиться» и сменить наклон.
- $\large s(t)$ — **сезонность**: сумма периодических волн — недельной, годовой, суточной.
- $\large h(t)$ — **праздники и события**: разовые поправки на конкретные даты из заданного списка.
- $\large \varepsilon_t$ — шум, который модель не объясняет.

#### Тренд: прямая с изломами
$$\LARGE g(t) = \Big(k + \sum_{j:\, s_j \le t} \delta_j\Big)\, t + \Big(m + \sum_{j:\, s_j \le t} \gamma_j\Big)$$

где:
- $\large k$ — начальный наклон тренда, $\large m$ — начальное смещение.
- $\large s_j$ — **точки излома** (changepoints): даты, в которых тренд может сменить наклон. По умолчанию — 25 штук, равномерно расставленных по первым 80% истории.
- $\large \delta_j$ — **изменение наклона** в точке $\large s_j$. После каждого пройденного излома наклон становится $\large k + \delta_1 + \ldots + \delta_j$.
- $\large \gamma_j$ — поправка смещения, которая подбирается автоматически, чтобы прямая в точке излома не «рвалась».

Ключевой момент: Prophet ставит изломы **всюду**, а потом штрафует их. На $\large \delta_j$ наложено априорное распределение Лапласа с масштабом `changepoint_prior_scale`: большинство $\large \delta_j$ сжимаются в 0, «выживают» только изломы, которые реально нужны данным. Маленький масштаб — тренд почти прямой, большой — тренд гибкий и гоняется за каждым поворотом.

#### Сезонность: ряд Фурье
$$\LARGE s(t) = \sum_{n=1}^{N} \left( a_n \cos\frac{2\pi n t}{P} + b_n \sin\frac{2\pi n t}{P} \right)$$

где:
- $\large P$ — период сезонности в днях: 7 для недельной, 365.25 для годовой.
- $\large N$ — порядок ряда Фурье (`fourier_order`): сколько гармоник складываем. Больше $\large N$ — форма волны сложнее и острее, но выше риск переобучения. По умолчанию: недельная $\large N=3$, годовая $\large N=10$.
- $\large a_n, b_n$ — коэффициенты, которые подбираются регрессией.

Недельная волна с $\large N=3$ — это 6 коэффициентов на 7 дней недели, то есть почти «отдельная поправка на каждый день», как у сезонного индекса `HoltWintersModel`. Но годовая волна — 20 коэффициентов на 365 дней: гладкая кривая вместо 365 отдельных поправок.

#### Как модель обучается и прогнозирует
- Все параметры ($\large k, m, \delta, a_n, b_n$, эффекты праздников) подбираются **одной задачей оптимизации** — MAP-оценка в Stan (через `cmdstanpy`). Отсюда служебные логи `cmdstanpy - INFO - Chain [1] ...` при каждом `fit()`.
- Прогноз на дату $\large t$ — просто значение $\large g(t) + s(t) + h(t)$ в этой точке. **Прогноз не рекурсивный**: модель не подставляет свои прогнозы вместо факта, и прогноз на 10-й день не зависит от прогноза на 9-й. Тренд за пределами истории — продолжение **последнего** отрезка прямой.

#### Отличие от
- `HoltWintersModel` тоже раскладывает ряд на уровень / тренд / сезонность, но обновляет их **рекурсивно**, шаг за шагом, с экспоненциальным забыванием прошлого (α/β/γ). Prophet подгоняет всю историю **сразу одной кривой**: свежие точки не весят больше старых, зато смена тренда описывается явно — изломом в конкретную дату.
- `SARIMAXModel` моделирует зависимость значения от прошлых значений и ошибок (AR/MA) и требует стационарности. Prophet никакой авторегрессии не содержит: в нём нет лагов, только функции времени. Поэтому стационарность ему не нужна, пропуски в истории не мешают, а неравномерные даты допустимы.

### 1.2 Параметры и реализация

Исходник `etna/models/prophet.py`, класс `ProphetModel` (сигнатура без сокращений — все параметры пробрасываются в `prophet.Prophet` один в один):

```python
class ProphetModel(
    PerSegmentModelMixin, PredictionIntervalContextIgnorantModelMixin, PredictionIntervalContextIgnorantAbstractModel
):
    def __init__(
        self,
        growth: str = "linear",
        changepoints: Optional[List[datetime]] = None,
        n_changepoints: int = 25,
        changepoint_range: float = 0.8,
        yearly_seasonality: Union[str, bool] = "auto",
        weekly_seasonality: Union[str, bool] = "auto",
        daily_seasonality: Union[str, bool] = "auto",
        holidays: Optional[pd.DataFrame] = None,
        seasonality_mode: str = "additive",
        seasonality_prior_scale: float = 10.0,
        holidays_prior_scale: float = 10.0,
        changepoint_prior_scale: float = 0.05,
        mcmc_samples: int = 0,
        interval_width: float = 0.8,
        uncertainty_samples: Union[int, bool] = 1000,
        stan_backend: Optional[str] = None,
        additional_seasonality_params: Iterable[Dict[str, Union[str, float, int]]] = (),
        timestamp_column: Optional[str] = None,
    ):
```

**Параметры `ProphetModel`:**

*Тренд $g(t)$:*
- `growth` *(по умолчанию `"linear"`)* — тип тренда: `"linear"` — кусочно-линейный; `"flat"` — тренда нет, константа; `"logistic"` — рост с насыщением к потолку `cap` (нужна колонка `cap` в данных как регрессор).
- `n_changepoints` *(25)* / `changepoint_range` *(0.8)* — сколько кандидатов в изломы и на какой доле истории их расставлять (первые 80%).
- `changepoints` — явный список дат изломов вместо автоматической расстановки.
- `changepoint_prior_scale` *(0.05)* — **гибкость тренда**: масштаб априорного распределения на $\delta_j$. Главный параметр для настройки.

*Сезонность $s(t)$:*
- `weekly_seasonality` / `yearly_seasonality` / `daily_seasonality` *(`"auto"`)* — `True`/`False`, число (= `fourier_order`) или `"auto"`. Правила `"auto"` из исходника `prophet`: недельная включается при истории ≥ 2 недель и шаге < 7 дней; годовая — при истории **≥ 730 дней (2 года)**; суточная — только при шаге < 1 дня.
- `seasonality_mode` *(`"additive"`)* — `"additive"`: $y = g + s$, или `"multiplicative"`: $y = g \cdot (1 + s)$ — размах волны пропорционален тренду.
- `seasonality_prior_scale` *(10.0)* — сила сезонности: меньше → волны сглаживаются к нулю.
- `additional_seasonality_params` — свои сезонности: список словарей для `Prophet.add_seasonality(name=..., period=..., fourier_order=...)`, например месячная `period=30.5`.

*Праздники $h(t)$:*
- `holidays` — датафрейм с колонками `holiday`, `ds` и опционально `lower_window` / `upper_window` (сколько дней до/после тоже считать праздником).
- `holidays_prior_scale` *(10.0)* — сила эффекта праздников.

*Остальное:*
- `mcmc_samples` *(0)* — 0: точечная MAP-оценка (быстро); > 0: полный байесовский вывод (медленно).
- `interval_width` / `uncertainty_samples` — ширина и число симуляций для интервалов прогноза.
- `timestamp_column` — брать время не из индекса, а из колонки-регрессора.

**В отличие от `HoltWintersModel`:**
- внутри — `prophet.Prophet`, а не `statsmodels`; обучение — оптимизация в Stan;
- **экзогенные регрессоры поддерживаются**: колонки, объявленные в `TSDataset` как `known_future`, ETNA сама передаёт в `Prophet.add_regressor()`. Колонки, неизвестные в будущем, игнорируются с предупреждением;
- **поддерживаются интервалы прогноза**: `model.forecast(future_ts, prediction_interval=True, quantiles=[0.025, 0.975])`;
- нет коэффициентов сглаживания: вместо α/β/γ — априорные масштабы `*_prior_scale`, которые задают не «скорость забывания», а «сколько свободы дать компоненте»;
- то же самое общее: `PerSegmentModelMixin` — отдельная модель на каждый сегмент; API `fit` → `make_future(HORIZON)` → `forecast`; `get_model()` → `{segment: prophet.Prophet}`; `forecast(..., return_components=True)` даёт разложение `target_component_trend`, `target_component_weekly`, `target_component_yearly`, ...

### 1.3 Особенности

- **Изломы только в первых 80% истории** (`changepoint_range=0.8`). Если тренд переломился в последние 20% train — а это самая важная часть для прогноза, — Prophet излом не увидит и продолжит старый наклон. Лечится `changepoint_range=0.9..0.95` или явным списком `changepoints`.
- **Прогноз тренда — продолжение последнего отрезка.** На длинном горизонте прямая уходит в небо или в ноль, как незатухающий тренд у `HoltWintersModel`. Для рядов без направления есть `growth="flat"`.
- **Нет рекурсии — нет «золотого правила».** В отличие от `MovingAverageModel` / `SeasonalMovingAverageModel`, прогноз Prophet на день `t+10` не опирается на прогноз дня `t+9`. Поэтому прогноз на 14 дней — это ровно первые 14 точек прогноза на 28 дней.
- **Годовая сезонность включается автоматически только при истории ≥ 2 лет.** На наших стандартных 5 месяцах (с 2026-04-01) её не будет — модель видит только тренд + недельную волну. Чтобы Prophet показал свою сильную сторону, нужна длинная история.
- **Разложение на компоненты работает только для аддитивного режима.** С `seasonality_mode="multiplicative"` вызов `forecast(..., return_components=True)` в ETNA падает с `ValueError: Forecast decomposition is only supported for additive components!` — компоненты тогда смотрим через `model.get_model()[segment].predict(...)` из самого `prophet`.
- **Шумные логи.** Каждый `fit()` пишет в лог `cmdstanpy`; на 10 юнитах × 5 фолдов это сотни строк. Отключаются через `logging.getLogger("cmdstanpy").disabled = True` (уже в ячейке импортов).
- **Праздники** — главная фишка Prophet, но в этой главе не используем: в тестовом окне 2026-08-18 … 2026-08-31 праздников нет. Вернёмся к ним, когда дойдём до регрессоров.

### 1.4 Перекрёстная проверка: `Pipeline.backtest`

До сих пор мы оценивали модели на **одном** тестовом окне — последних 14 днях. Это одна точка: если именно эти две недели оказались необычными, вывод «модель A лучше B» может оказаться случайным. **Перекрёстная проверка на временных рядах** (backtest) повторяет «обучи → спрогнозируй → посчитай метрику» несколько раз, сдвигая тестовое окно назад по истории, и смотрит на ошибку по всем окнам.

```
expand (расширяющееся окно)              constant (скользящее окно)
fold 0: [train====]  [test]              fold 0: [train====]  [test]
fold 1: [train========]  [test]          fold 1:     [train====]  [test]
fold 2: [train============]  [test]      fold 2:         [train====]  [test]
```

- **`mode="expand"`** *(по умолчанию)* — начало train всегда одно и то же, каждый следующий фолд добавляет историю. Так модель живёт в проде: переобучается на всём, что накопилось.
- **`mode="constant"`** — train одной длины, окно скользит. Полезно, когда старая история вредит (ряд поменял поведение) или чтобы сравнивать фолды «при равных условиях».
- Тестовые окна идут **подряд и не пересекаются**: последний фолд — это ровно наш обычный holdout (последние `horizon` точек), предыдущие — `horizon` точек перед ним и т. д.

`backtest` запускается не у модели, а у **конвейера** `Pipeline(model, transforms, horizon)`: преобразования (их добавим дальше, вместе с конструированием признаков) должны обучаться заново на каждом фолде, иначе информация из теста просочится в train.

Исходник `etna/pipeline/base.py`, метод `BasePipeline.backtest` (ETNA 3.0.0):

```python
def backtest(
    self,
    ts: TSDataset,
    metrics: List[BaseMetric],
    n_folds: Union[int, List[FoldMask]] = 5,
    mode: Optional[str] = None,
    aggregate_metrics: bool = False,
    n_jobs: int = 1,
    refit: Union[bool, int] = True,
    stride: Optional[int] = None,
    joblib_params: Optional[Dict[str, Any]] = None,
    forecast_params: Optional[Dict[str, Any]] = None,
) -> Dict[str, Union[pd.DataFrame, List[TSDataset], List[Self]]]:
```

**Параметры:**
- `ts` — **весь** ряд (train + test): фолды нарезаются внутри;
- `metrics` — список метрик, например `[SMAPE(), MAE()]`;
- `n_folds` *(5)* — число фолдов, либо явный список `FoldMask`;
- `mode` — `"expand"` / `"constant"`;
- `aggregate_metrics` *(False)* — `True`: одна строка на сегмент со средним по фолдам; `False`: строка на каждую пару (сегмент, фолд);
- `refit` *(True)* — переобучать модель на каждом фолде; `False` — обучить один раз на первом; число `k` — переобучать каждые `k` фолдов;
- `stride` — шаг между фолдами, по умолчанию = `horizon` (окна встык);
- `n_jobs` — параллельно по фолдам; `joblib_params=dict(verbose=0)` убирает прогресс-лог joblib.

**Важное расхождение с пособием.** В пособии (ETNA 2.x) `backtest` возвращает кортеж из трёх датафреймов: `metrics_df, forecast_df, fold_info_df = pipeline.backtest(...)`. В ETNA 3.0.0 он возвращает **словарь**:
- `result["metrics"]` — метрики: сегмент × фолд (или сегмент, если `aggregate_metrics=True`);
- `result["forecasts"]` — **список** `TSDataset`, по одному прогнозу на фолд (а не один широкий датафрейм);
- `result["fold_info"]` — даты train/test каждого фолда;
- `result["pipelines"]` — обученные конвейеры каждого фолда.

Соответственно и `plot_backtest(result["forecasts"], ts, history_len=...)` принимает список прогнозов.

## 2. Данные

Те же файлы, что и в предыдущих тетрадях (`local/data/`, в `.gitignore`):
- `units.csv` — 10 юнитов; для основной практики берём 2 из них (размер `10+`, typical + busier), для бонуса — все 10.
- `facts-by-date.csv` — дневные факты, 2022-01-01…2026-08-31.

Период и сплит — как в прошлых тетрадях: с 2026-04-01, test — последние 14 дней (2026-08-18 … 2026-08-31). Тогда результаты сравнимы с `NaiveModel` / `MovingAverageModel` / `SeasonalMovingAverageModel` / `SARIMAXModel` / `HoltWintersModel` напрямую, в том числе в MLflow.

Новое в этой главе — **длинная история**: у обоих юнитов `10+` факты есть с 2022 года, этого хватает, чтобы Prophet включил годовую сезонность (Задание 6). Тестовое окно при этом то же самое, меняется только длина train.

В этой главе не сравниваем с продовым прогнозом (`forecasts-by-date.csv`) — это отдельная тема, которой займёшься сам.

In [ ]:
import logging
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etna.datasets import TSDataset
from etna.models import ProphetModel, HoltWintersModel
from etna.pipeline import Pipeline
from etna.metrics import SMAPE, WAPE, MAPE, MAE
from etna.analysis import plot_forecast, plot_backtest

from src.custom_logger import log_to_mlflow
from src.prod_metrics import bias, pooled

# Prophet пишет в лог cmdstanpy на каждом fit() — на 10 юнитах × 5 фолдах это сотни строк
logging.getLogger("cmdstanpy").disabled = True
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = "../../local/data"
HORIZON = 14

## 3. Практика

От простого к сложному: на синтетике смотрим, чем Prophet отличается от `HoltWintersModel` (излом тренда), разбираем прогноз на компоненты и знакомимся с `backtest`. Потом — 2 реальных юнита на стандартном и на длинном периоде, и в бонусе — все 10 юнитов с логированием в MLflow.

### Задание 1. Синтетика: излом тренда

1. Собери синтетический ряд на 18 недель (126 дней): **кусочно-линейный тренд** — растёт на +2 в день до 70-го дня, потом **падает** на −1 в день (подсказка: `np.where(t < 70, ..., ...)`), — плюс недельный паттерн и немного шума (`np.random.normal(0, 3, ...)`), как в синтетике для `HoltWintersModel`. Оберни в `TSDataset` с одним сегментом `"synthetic"`, раздели на train/test (последние 14 дней).
2. Обучи `ProphetModel()` с параметрами по умолчанию и `HoltWintersModel(trend="add", seasonal="add", seasonal_periods=7)`. Нарисуй оба прогноза на одном графике (`plot_forecast` со словарём) и посчитай SMAPE.
3. Кто лучше справился с изломом и почему? Вспомни раздел 1.1: как каждая из моделей «узнаёт», что наклон сменился?
4. Повтори Prophet с `changepoint_prior_scale` = `0.001`, `0.05` (по умолчанию), `0.5`. Как меняется прогноз и SMAPE? Что происходит при очень маленьком значении?
5. Посмотри, где Prophet нашёл излом: `model.get_model()["synthetic"]` — это объект `prophet.Prophet`; даты кандидатов лежат в `.changepoints`, изменения наклона — в `.params["delta"][0]`. Построй `delta` против даты: какой кандидат «выжил»? Совпадает ли он с 70-м днём?

### Задание 2. Из чего собран прогноз

1. Для `ProphetModel()` из Задания 1 получи прогноз с `return_components=True` и достань компоненты через `get_target_components()`. Какие компоненты есть? Почему нет `yearly`?
2. Проверь, что `trend + weekly` в сумме дают `target`.
3. Сравни 7 значений компоненты `weekly` с недельным паттерном, из которого собиралась синтетика (минус его среднее). Насколько точно ряд Фурье с 3 гармониками восстановил узор?
4. Нарисуй компоненту `trend` прогноза вместе с трендом, из которого ты собирал синтетику. Это прямая? С каким наклоном — с последним (−1) или каким-то другим?
5. **Нерекурсивность.** Сделай прогноз той же обученной моделью на 28 дней (`make_future(28)`) и сравни первые 14 точек с прогнозом на 14 дней. Совпадают? Почему для `MovingAverageModel` это было бы не так (раздел 1.3)?

### Задание 3. Синтетика: перекрёстная проверка `expand` vs `constant`

1. Собери `Pipeline(model=ProphetModel(), transforms=[], horizon=HORIZON)` и запусти `backtest` на **всём** синтетическом ряде из Задания 1 (не на train!) с `metrics=[SMAPE()]`, `n_folds=4`, `mode="expand"`. Помни: в ETNA 3.0.0 результат — словарь (раздел 1.4).
2. Посмотри `result["fold_info"]`: какие даты у train и test каждого фолда? На каком фолде тестовое окно совпадает с holdout из Задания 1 — и совпадает ли SMAPE на нём?
3. Посмотри `result["metrics"]`: на каком фолде ошибка самая большая? Сопоставь даты этого фолда с 70-м днём — что модель «знала» об изломе в момент прогноза?
4. Визуализируй `plot_backtest(result["forecasts"], ts)` и с `history_len="all"`.
5. Повтори с `mode="constant"` и сравни `fold_info` и метрики. Чем отличается train у фолдов? Где разница в SMAPE заметнее и почему?
6. Повтори `expand` с `aggregate_metrics=True` — что теперь в `result["metrics"]`?

### Задание 4. Загрузи данные по 2 юнитам размера `10+`

1. Прочитай `facts-by-date.csv` и `units.csv`, оставь только 2 юнита размера `10+` (typical + busier) с 2026-04-01 — так же, как для предыдущих моделей.
2. Собери `TSDataset` (дневная частота), где `segment` = `MaskedUnitId`.
3. Раздели на train/test (последние `HORIZON` = 14 дней).
4. Прежде чем обучать: по графикам из тетради про `HoltWintersModel` ты уже знаешь, что тренда у этих рядов почти нет, а недельная «пила» сильная. Какие параметры Prophet из раздела 1.2 это подсказывает попробовать?

### Задание 5. Prophet на двух юнитах: holdout и backtest

1. Обучи на обоих сегментах несколько конфигураций:
   - `ProphetModel()` — по умолчанию;
   - `ProphetModel(growth="flat")` — без тренда;
   - `ProphetModel(seasonality_mode="multiplicative")`;
   - `ProphetModel(changepoint_prior_scale=0.5)` и `ProphetModel(changepoint_prior_scale=0.005)` — гибкий и жёсткий тренд.
2. Посчитай `SMAPE` per-segment и `mode="macro"`, собери в одну таблицу (строки — конфигурации, колонки — юниты + macro). Для ориентира — на тех же 2 юнитах и 14 днях `HoltWintersModel(trend=None, seasonal="add")` давал macro 7.77, лучший `SARIMAXModel` — 7.42.
3. Для лучшей конфигурации: `plot_forecast` и разложение на компоненты (`return_components=True`). Как выглядит тренд в прогнозе — горизонтальный, растёт, падает?
4. **Holdout — это одно окно.** Прогони через `backtest` (`n_folds=5`, `mode="expand"`, `aggregate_metrics=True`) три конвейера: лучший Prophet, Prophet по умолчанию и `HoltWintersModel(trend=None, seasonal="add", seasonal_periods=7)`. Сохраняется ли порядок моделей, который был на holdout? Насколько разброс SMAPE между фолдами (`aggregate_metrics=False`) больше разницы между моделями?
5. Сделай вывод: можно ли по одному тестовому окну в 14 дней уверенно выбирать между моделями с разницей в 0.5–1 SMAPE?

### Задание 6. Длинная история: годовая сезонность

У юнитов `10+` факты есть с 2022 года. Проверим, что даст Prophet длинная история: при train ≥ 2 лет он сам включает годовую сезонность (раздел 1.2).

1. Собери второй `TSDataset` по тем же 2 юнитам, но **с 2024-01-01**. Сплит тот же: последние 14 дней — test, то есть тестовое окно совпадает с Заданием 5.
2. Построй `ts.plot()` на длинной истории. Видна ли годовая волна? Меняется ли уровень от года к году — и как при этом ведёт себя ширина недельной «пилы»?
3. Обучи `ProphetModel()` (аддитивная сезонность) и `ProphetModel(seasonality_mode="multiplicative")`. Убедись, что годовая сезонность включилась: `model.get_model()[segment].seasonalities`.
4. Сравни SMAPE с короткой историей из Задания 5 — таблица 2 × 2 (история × режим сезонности). Где длинная история помогла, а где навредила?
5. Объясни результат по таксономии add/mul из тетради про `HoltWintersModel`: что случается с аддитивной недельной волной постоянного размаха, когда уровень ряда за 2.5 года сильно менялся?
6. Посмотри на годовую компоненту: для аддитивной модели — через `return_components=True`, для мультипликативной (раздел 1.3: в ETNA разложение не работает) — через `prophet` напрямую: `m = model.get_model()[segment]`, `m.plot_components(m.predict(m.make_future_dataframe(HORIZON)))`. Где на годовой кривой сейчас конец августа — на подъёме или на спаде?

### Бонус. Общая картина по всем 10 юнитам + логирование

1. Прогони 2–3 лучшие конфигурации из Заданий 5–6 сразу на всех 10 юнитах на стандартном периоде (с 2026-04-01, без фильтра по `unit_size`).
2. Посчитай весь набор метрик на holdout — `SMAPE`, `WAPE`, `MAPE`, `MAE`, `Bias` (через `src/prod_metrics.py`), per-segment и агрегированно (`smape_macro`, `mape_macro`, `mae_macro`, `wape_pooled`, `bias_pooled`, `fit_sec`) — те же имена, что у прошлых запусков.
3. **Новое — метрика по backtest.** Для каждой конфигурации прогони `backtest` (`n_folds=5`, `mode="expand"`, `aggregate_metrics=True`) и добавь к агрегатам `smape_backtest_macro` — среднее SMAPE по юнитам и 5 фолдам. Это более устойчивая оценка, чем 14 дней holdout. Для сравнения прогони так же лучшую `HoltWintersModel` и залогируй её отдельным запуском, чтобы у прошлой модели тоже была эта метрика.
4. Собери таблицу `result` по юнитам: `segment` / `unit_size` / `pick_reason` / `total_orders` + все метрики + SMAPE по backtest. На каких юнитах Prophet выигрывает у `HoltWintersModel`, а на каких проигрывает?
5. Залогируй каждую конфигурацию отдельным запуском через `log_to_mlflow(...)` — по аналогии с бонусом `HoltWintersModel`:
   - имя запуска: `Prophet_<вариант>_ch07_all_10_day`, например `Prophet_default_ch07_all_10_day`, `Prophet_mult_ch07_all_10_day`;
   - теги: `{"chapter": "ch07", "units_scope": "all_10", "granularity": "day", "model_family": "prophet", "approach": "one_config", "impl": "prophet"}` + теги варианта (`"growth"`, `"seasonality_mode"`, `"changepoint_prior_scale"`);
   - параметры: параметры модели + `horizon`, а для backtest-метрики — `backtest_n_folds` и `backtest_mode`.
6. *(По желанию)* Вариант с длинной историей на всех 10 юнитах. Осторожно: у части юнитов до открытия в данных стоят нули (`8a399130` и `a5cd0472` открылись только в марте 2026). Обрежь каждый ряд по первому ненулевому факту, а при логировании добавь тег `"history": "full"` — тестовое окно то же, поэтому метрики сравнимы, но train разной длины.
7. Сравни в MLflow (или через `mlflow.search_runs`, как в конце тетради про `HoltWintersModel`) лучшие Prophet с `HoltWinters_*` и `SARIMAX_*` по юнитам.